# Primetrade.ai — Trader Performance vs Market Sentiment
**Data Science Intern Assignment — Round 0 | REAL DATA**

| | |
|---|---|
| **Datasets** | `fear_greed_index.csv` × `historical_data_csv.gz` |
| **Window** | May 2023 – May 2025 (731 days) |
| **Trades** | 211,224 across 32 accounts and 246 coins |
| **Sentiment days** | Fear: 144 \| Neutral: 177 \| Greed: 410 |

---
## Table of Contents
1. [Setup & Imports](#setup)
2. [Part A — Data Preparation](#parta)
   - A1. Load & quality check
   - A2. Timestamp alignment & merge
   - A3. Feature engineering
3. [Part B — Analysis](#partb)
   - B1. Performance: Fear vs Greed
   - B2. Behavioral shifts
   - B3. Trader segmentation
4. [Part C — Strategy Recommendations](#partc)
5. [Bonus — Predictive model + Clustering](#bonus)
---

<a id='setup'></a>
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# ── Palette ──
FEAR_COLOR    = '#E63946'
GREED_COLOR   = '#2EC4B6'
NEUTRAL_COLOR = '#F4A261'
EF_COLOR      = '#9B1D20'
EG_COLOR      = '#1A7A74'
BG, CARD, TEXT, GRID, ACCENT = '#0D1117','#161B22','#E6EDF3','#30363D','#A8DADC'
BIN_COLORS  = {'Fear': FEAR_COLOR, 'Greed': GREED_COLOR, 'Neutral': NEUTRAL_COLOR}
SENT_COLORS = {'Extreme Fear': EF_COLOR, 'Fear': FEAR_COLOR, 'Neutral': NEUTRAL_COLOR,
               'Greed': GREED_COLOR, 'Extreme Greed': EG_COLOR}

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': GRID, 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': GRID,
    'grid.linestyle': '--', 'grid.alpha': 0.4,
    'legend.facecolor': CARD, 'legend.edgecolor': GRID,
})
np.random.seed(42)
print('Setup complete ✅')

<a id='parta'></a>
## Part A — Data Preparation
### A1. Load & Quality Check

In [ ]:
# ── Load Fear/Greed ──────────────────────────────────────────────────────────
fg_raw = pd.read_csv('data/fear_greed_index.csv')
fg_raw['date'] = pd.to_datetime(fg_raw['date'])
fg_raw = fg_raw.sort_values('date').drop_duplicates('date')
fg_raw['sentiment'] = fg_raw['classification'].apply(
    lambda x: 'Fear' if 'Fear' in x else ('Greed' if 'Greed' in x else 'Neutral'))

# ── Load Trader Data ─────────────────────────────────────────────────────────
td_raw = pd.read_csv('data/historical_data_csv.gz', compression='gzip')
td_raw['datetime'] = pd.to_datetime(td_raw['Timestamp IST'], format='%d-%m-%Y %H:%M', dayfirst=True)
td_raw['date'] = td_raw['datetime'].dt.normalize()

print(f'Fear/Greed : {fg_raw.shape[0]:,} rows × {fg_raw.shape[1]} cols')
print(f'             {fg_raw["date"].min().date()} → {fg_raw["date"].max().date()}')
print(f'Trader data: {td_raw.shape[0]:,} rows × {td_raw.shape[1]} cols')
print(f'             {td_raw["date"].min().date()} → {td_raw["date"].max().date()}')
print(f'             {td_raw["Account"].nunique()} unique accounts | {td_raw["Coin"].nunique()} unique coins')

In [ ]:
print('=== Missing Values — Fear/Greed ===')
print(fg_raw.isnull().sum())
print(f'Duplicates: {fg_raw.duplicated().sum()}')
print('\n=== Missing Values — Trader Data ===')
print(td_raw.isnull().sum())
print(f'Duplicates: {td_raw.duplicated().sum()}')

In [ ]:
print('=== Trader Data dtypes & sample ===')
print(td_raw.dtypes)
td_raw.head(3)

In [ ]:
print('=== Fear/Greed Classification Distribution (overlap window) ===')
fg_raw['classification'].value_counts()

In [ ]:
print('=== Top 10 Coins by Trade Count ===')
print(td_raw['Coin'].value_counts().head(10).to_string())
print('\n=== Direction Distribution ===')
print(td_raw['Direction'].value_counts().to_string())
print('\n=== Closed PnL describe ===')
td_raw['Closed PnL'].describe().round(2)

### A2. Timestamp Alignment & Merge

In [ ]:
# Filter to overlap window
overlap_start = max(td_raw['date'].min(), fg_raw['date'].min())
overlap_end   = min(td_raw['date'].max(), fg_raw['date'].max())

fg = fg_raw[(fg_raw['date'] >= overlap_start) & (fg_raw['date'] <= overlap_end)].copy()
td = td_raw[(td_raw['date'] >= overlap_start) & (td_raw['date'] <= overlap_end)].copy()

print(f'Overlap window: {overlap_start.date()} → {overlap_end.date()} ({(overlap_end-overlap_start).days} days)')
print(f'FG rows in window: {fg.shape[0]:,}')
print(f'TD rows in window: {td.shape[0]:,}')

merged = td.merge(
    fg[['date','value','classification','sentiment']].rename(columns={'value':'fg_value'}),
    on='date', how='left'
)
unmatched = merged['sentiment'].isna().sum()
print(f'\nMerged shape: {merged.shape}')
print(f'Unmatched rows (no FG for that date): {unmatched} ({unmatched/len(merged):.4%})')

# Drop the 6 unmatched rows
merged = merged.dropna(subset=['sentiment'])
print(f'Final merged shape: {merged.shape}')

### A3. Feature Engineering — Key Metrics

In [ ]:
# Identify closing trades
close_dirs = {'Close Long','Close Short','Long > Short','Short > Long',
              'Liquidated Isolated Short','Liquidated Isolated Long','Auto-Deleveraging'}
merged['is_close'] = merged['Direction'].isin(close_dirs)
merged['is_long']  = (merged['Side'] == 'BUY')
merged['pnl_net']  = merged['Closed PnL'] - merged['Fee']
merged['is_win']   = (merged['Closed PnL'] > 0).astype(int)

# Daily per-account aggregation
def daily_agg(df):
    closes = df[df['is_close'] | (df['Closed PnL'] != 0)]
    return pd.Series({
        'n_trades'         : len(df),
        'n_closes'         : len(closes),
        'daily_pnl'        : df['pnl_net'].sum(),
        'realized_pnl'     : closes['Closed PnL'].sum(),
        'total_fees'       : df['Fee'].sum(),
        'avg_size_usd'     : df['Size USD'].mean(),
        'total_vol_usd'    : df['Size USD'].sum(),
        'wins'             : (closes['Closed PnL'] > 0).sum(),
        'n_longs'          : df['is_long'].sum(),
        'n_shorts'         : (~df['is_long']).sum(),
        'cross_margin_pct' : df['Crossed'].mean(),
        'n_liquidations'   : (df['Direction'].str.contains('Liquidat', na=False)).sum(),
    })

daily = (merged.groupby(['date','Account','sentiment','classification','fg_value'])
               .apply(daily_agg, include_groups=False)
               .reset_index())

daily['win_rate']      = daily['wins'] / daily['n_closes'].replace(0, np.nan)
daily['long_ratio']    = daily['n_longs'] / (daily['n_trades'] + 1e-9)
daily['is_day_winner'] = (daily['daily_pnl'] > 0).astype(int)

print(f'Daily records: {daily.shape[0]:,}  (account × date rows)')
daily[['daily_pnl','n_trades','win_rate','long_ratio','cross_margin_pct']].describe().round(3)

In [ ]:
# Account-level lifetime stats
def account_agg(df):
    closes = df[df['is_close'] | (df['Closed PnL'] != 0)]
    pnl_cs = closes.sort_values('datetime')['Closed PnL'].cumsum()
    dd = (pnl_cs - pnl_cs.cummax()).min() if len(pnl_cs) else 0
    return pd.Series({
        'total_pnl'        : df['pnl_net'].sum(),
        'realized_pnl'     : closes['Closed PnL'].sum(),
        'total_fees'       : df['Fee'].sum(),
        'n_trades'         : len(df),
        'n_closes'         : len(closes),
        'win_rate'         : (closes['Closed PnL'] > 0).mean() if len(closes) else np.nan,
        'avg_size_usd'     : df['Size USD'].mean(),
        'total_vol_usd'    : df['Size USD'].sum(),
        'cross_margin_pct' : df['Crossed'].mean(),
        'long_ratio'       : df['is_long'].mean(),
        'max_drawdown'     : dd,
        'n_liquidations'   : (df['Direction'].str.contains('Liquidat', na=False)).sum(),
        'n_coins'          : df['Coin'].nunique(),
        'active_days'      : df['date'].nunique(),
    })

acct = (merged.groupby('Account')
              .apply(account_agg, include_groups=False)
              .reset_index())

acct['cross_segment'] = acct['cross_margin_pct'].apply(
    lambda x: 'High Cross (>70%)' if x > 0.7 else ('Low Cross (<30%)' if x < 0.3 else 'Mixed'))
acct['freq_segment'] = pd.qcut(acct['n_trades'], q=3,
                                labels=['Infrequent','Moderate','Frequent'])
acct['perf_segment'] = acct['total_pnl'].apply(
    lambda x: 'Consistent Winner' if x > 0 else 'Consistent Loser')

print(f'Accounts: {acct.shape[0]}')
print(f'Profitable: {(acct["total_pnl"]>0).sum()} / {len(acct)}')
acct[['total_pnl','win_rate','n_trades','max_drawdown','total_fees']].describe().round(2)

---
<a id='partb'></a>
## Part B — Analysis
### B1. Performance: Fear vs Greed Days

In [ ]:
sent_perf = daily.groupby('sentiment').agg(
    n_trader_days    = ('daily_pnl','count'),
    avg_daily_pnl    = ('daily_pnl','mean'),
    median_daily_pnl = ('daily_pnl','median'),
    win_rate         = ('win_rate','mean'),
    avg_n_trades     = ('n_trades','mean'),
    avg_vol_usd      = ('total_vol_usd','mean'),
    avg_long_ratio   = ('long_ratio','mean'),
).round(2)

fear_pnl  = daily.loc[daily['sentiment']=='Fear',  'daily_pnl']
greed_pnl = daily.loc[daily['sentiment']=='Greed', 'daily_pnl']
u, p = stats.mannwhitneyu(fear_pnl, greed_pnl, alternative='two-sided')

print('=== Performance Table by Sentiment ===')
print(sent_perf.to_string())
print(f'\nMann-Whitney U={u:.0f}, p={p:.6f} → Significant at 5%: {p < 0.05}')

In [ ]:
order5 = ['Extreme Fear','Fear','Neutral','Greed','Extreme Greed']
cls_perf = daily.groupby('classification').agg(
    avg_daily_pnl = ('daily_pnl','mean'),
    win_rate      = ('win_rate','mean'),
    avg_n_trades  = ('n_trades','mean'),
    n_days        = ('daily_pnl','count'),
).reindex(order5).round(2)
print('=== 5-Class Performance ===')
cls_perf

In [ ]:
fig = plt.figure(figsize=(16, 6), facecolor=BG)
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])

for s, col in [('Fear', FEAR_COLOR), ('Greed', GREED_COLOR), ('Neutral', NEUTRAL_COLOR)]:
    data = daily.loc[daily['sentiment']==s, 'daily_pnl']
    ax1.hist(data.clip(-5000, 5000), bins=60, alpha=0.72, color=col,
             label=f"{s}  (μ=${data.mean():,.0f}  med=${data.median():,.0f})", edgecolor='none')
ax1.axvline(0, color='white', lw=1.2, ls='--', alpha=0.7)
ax1.set_xlabel('Daily Net PnL per Trader (USD)')
ax1.set_ylabel('Frequency')
ax1.set_title('PnL Distribution by Sentiment\n(Fear has fat right tail; Greed is tighter & more positive)', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True)

wr5 = daily.groupby('classification')['win_rate'].mean().reindex(order5)
cols5 = [SENT_COLORS[c] for c in order5]
bars = ax2.bar(range(len(order5)), wr5.values, color=cols5, edgecolor='none', width=0.6)
ax2.set_xticks(range(len(order5)))
ax2.set_xticklabels(['Ext.\nFear','Fear','Neutral','Greed','Ext.\nGreed'], fontsize=9)
ax2.axhline(0.5, color='white', lw=0.8, ls='--', alpha=0.7, label='50%')
ax2.set_ylabel('Avg Win Rate'); ax2.set_title('Win Rate by Classification', fontweight='bold')
ax2.set_ylim(0, 1.0); ax2.legend(); ax2.grid(True, axis='y')
for bar, val in zip(bars, wr5.values):
    if not np.isnan(val):
        ax2.text(bar.get_x()+bar.get_width()/2, val+0.02, f'{val:.1%}',
                 ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Chart 1 — Trader Performance: Fear vs Greed Days  (Real Hyperliquid Data)',
             fontsize=13, fontweight='bold', color=TEXT, y=1.01)
plt.savefig('charts/chart1_pnl_winrate.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 KEY FINDING: Fear days have the highest MEAN PnL ($5,037) but lowest MEDIAN ($104).\n'
      '   Greed days: mean $4,067 but median $236 — far more CONSISTENT returns.')

### B2. Behavioral Shifts by Sentiment

In [ ]:
behav = daily.groupby('sentiment').agg(
    avg_n_trades    = ('n_trades','mean'),
    avg_vol_usd     = ('total_vol_usd','mean'),
    avg_long_ratio  = ('long_ratio','mean'),
    cross_margin    = ('cross_margin_pct','mean'),
    avg_size_usd    = ('avg_size_usd','mean'),
).round(3)
print('=== Behavioral Metrics by Sentiment ===')
behav

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5), facecolor=BG)
fig.suptitle('Chart 2 — Trader Behavior Shifts by Sentiment', fontsize=14, fontweight='bold', color=TEXT, y=1.01)

order3  = ['Fear','Neutral','Greed']
cols3   = [BIN_COLORS[s] for s in order3]
metrics = [
    ('avg_n_trades',   'Avg Trades / Day',       False),
    ('avg_vol_usd',    'Avg Daily Volume (USD)',  False),
    ('avg_long_ratio', 'Avg Long Ratio',          True),
    ('cross_margin',   'Avg Cross-Margin %',      True),
]
for ax, (col, label, pct) in zip(axes, metrics):
    vals = behav.loc[order3, col]
    bars = ax.bar(order3, vals.values, color=cols3, edgecolor='none', width=0.5)
    ax.set_title(label, fontweight='bold'); ax.grid(True, axis='y')
    if pct:
        ax.axhline(0.5, color='white', lw=0.8, ls='--', alpha=0.6)
        ax.set_ylim(0, 0.85)
    for bar, val in zip(bars, vals.values):
        fmt = f'{val:.1%}' if pct else f'{val:,.0f}'
        ax.text(bar.get_x()+bar.get_width()/2, val + vals.max()*0.02,
                fmt, ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart2_behavior_shifts.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 KEY FINDING: Traders place 37% MORE trades and 115% MORE volume during Fear vs Greed.\n'
      '   Long ratio is HIGHER during Fear (52.2%) — buy-the-dip contrarian behavior.')

### B3. Trader Segmentation (3 axes)

In [ ]:
# Segment 1: Cross-margin usage
seg1 = acct.groupby('cross_segment').agg(
    n_accounts    = ('Account','count'),
    avg_total_pnl = ('total_pnl','mean'),
    avg_win_rate  = ('win_rate','mean'),
    avg_drawdown  = ('max_drawdown','mean'),
).round(2)
print('=== Segment 1: Cross-Margin Usage ===')
print(seg1.to_string())

# Segment 2: Trade frequency
seg2 = acct.groupby('freq_segment', observed=True).agg(
    n_accounts    = ('Account','count'),
    avg_total_pnl = ('total_pnl','mean'),
    avg_win_rate  = ('win_rate','mean'),
).round(2)
print('\n=== Segment 2: Trade Frequency ===')
print(seg2.to_string())

# Segment 3: Profitable vs loss
seg3 = acct.groupby('perf_segment').agg(
    n_accounts    = ('Account','count'),
    avg_total_pnl = ('total_pnl','mean'),
    avg_win_rate  = ('win_rate','mean'),
    avg_n_trades  = ('n_trades','mean'),
).round(2)
print('\n=== Segment 3: Performance Consistency ===')
print(seg3.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('Chart 3 — Trader Segments: PnL, Win Rate & Drawdown', fontsize=14, fontweight='bold', color=TEXT, y=1.01)

# Scatter: all accounts
sc = axes[0].scatter(
    acct['win_rate'], acct['total_pnl'],
    s=np.clip(acct['total_vol_usd']/5e5, 30, 400),
    c=acct['total_pnl'], cmap='RdYlGn',
    alpha=0.85, edgecolors='none',
    vmin=acct['total_pnl'].quantile(0.05), vmax=acct['total_pnl'].quantile(0.95)
)
plt.colorbar(sc, ax=axes[0], label='Total PnL (USD)')
axes[0].axhline(0, color='white', lw=0.8, ls='--')
axes[0].axvline(0.5, color='white', lw=0.8, ls='--')
axes[0].set_xlabel('Win Rate'); axes[0].set_ylabel('Total PnL (USD)')
axes[0].set_title('All 32 Accounts  (size = volume)', fontweight='bold')
axes[0].grid(True)

# Cross-margin segment
csegs = acct.groupby('cross_segment')['total_pnl'].mean().sort_values()
bar_c = [GREED_COLOR if v >= 0 else FEAR_COLOR for v in csegs.values]
axes[1].barh(csegs.index, csegs.values, color=bar_c, edgecolor='none', height=0.45)
axes[1].axvline(0, color='white', lw=0.8, ls='--')
axes[1].set_xlabel('Avg Total PnL (USD)')
axes[1].set_title('Avg PnL by Cross-Margin Segment', fontweight='bold')
axes[1].grid(True, axis='x')
for i, (idx, val) in enumerate(csegs.items()):
    axes[1].text(val + abs(csegs).max()*0.02*(1 if val>=0 else -1),
                 i, f'${val:,.0f}', va='center', fontsize=9)

# Max drawdown vs PnL
axes[2].scatter(acct['max_drawdown'], acct['total_pnl'],
                c=[GREED_COLOR if p > 0 else FEAR_COLOR for p in acct['total_pnl']],
                s=80, alpha=0.85, edgecolors='none')
axes[2].axhline(0, color='white', lw=0.8, ls='--')
axes[2].set_xlabel('Max Drawdown (USD)'); axes[2].set_ylabel('Total PnL (USD)')
axes[2].set_title('Max Drawdown vs Total PnL', fontweight='bold'); axes[2].grid(True)

plt.tight_layout()
plt.savefig('charts/chart3_account_segments.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 KEY FINDING: High cross-margin users avg $1.29M total PnL vs $243K for low cross users.\n'
      '   Cross-margin = sophistication signal.')

In [ ]:
# Time-series chart
daily_mkt = daily.groupby(['date','fg_value','sentiment']).agg(
    total_pnl = ('daily_pnl','sum'),
).reset_index().sort_values('date')
daily_mkt['pnl_30d'] = daily_mkt['total_pnl'].rolling(30, min_periods=1).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8), facecolor=BG,
                                 gridspec_kw={'height_ratios':[2,1],'hspace':0.08})
fig.suptitle('Chart 4 — Fear/Greed Index vs Market PnL  (May 2023 – May 2025)',
             fontsize=13, fontweight='bold', color=TEXT, y=0.99)

ax1.plot(daily_mkt['date'], daily_mkt['fg_value'], color=NEUTRAL_COLOR, lw=1.4, label='FG Index')
ax1.fill_between(daily_mkt['date'], daily_mkt['fg_value'], 50,
                  where=daily_mkt['fg_value']>=50, alpha=0.18, color=GREED_COLOR, label='Greed zone')
ax1.fill_between(daily_mkt['date'], daily_mkt['fg_value'], 50,
                  where=daily_mkt['fg_value']<50,  alpha=0.18, color=FEAR_COLOR,  label='Fear zone')
ax1.axhline(50, color=GRID, lw=0.8); ax1.set_ylabel('FG Index')
ax1.set_ylim(0,100); ax1.legend(); ax1.grid(True); ax1.set_xticklabels([])

ax2b = ax2.twinx()
colors_bar = [BIN_COLORS.get(s, NEUTRAL_COLOR) for s in daily_mkt['sentiment']]
ax2.bar(daily_mkt['date'], daily_mkt['total_pnl'], color=colors_bar, alpha=0.4, width=0.8, label='Daily PnL')
ax2b.plot(daily_mkt['date'], daily_mkt['pnl_30d'], color=ACCENT, lw=2, label='30d MA')
ax2.set_ylabel('Daily PnL (USD)'); ax2b.set_ylabel('30d MA', color=ACCENT)
ax2b.tick_params(axis='y', labelcolor=ACCENT)
lines, lbls = ax2.get_legend_handles_labels(); lines2, lbls2 = ax2b.get_legend_handles_labels()
ax2.legend(lines+lines2, lbls+lbls2, loc='upper left'); ax2.grid(True)

plt.savefig('charts/chart4_timeseries.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Long ratio violin + top coins by sentiment
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), facecolor=BG)
fig.suptitle('Chart 5 — Long-Ratio Shift & Top Coin PnL by Sentiment',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

order3 = ['Fear','Neutral','Greed']
data_lr = [daily.loc[daily['sentiment']==s, 'long_ratio'].dropna() for s in order3]
vp = axes[0].violinplot(data_lr, showmedians=True)
for body, col in zip(vp['bodies'], [FEAR_COLOR, NEUTRAL_COLOR, GREED_COLOR]):
    body.set_facecolor(col); body.set_alpha(0.75)
vp['cmedians'].set_color('white'); vp['cbars'].set_color(GRID)
vp['cmins'].set_color(GRID); vp['cmaxes'].set_color(GRID)
axes[0].set_xticks([1,2,3]); axes[0].set_xticklabels(order3)
axes[0].axhline(0.5, color='white', lw=0.8, ls='--', label='50% neutral')
axes[0].set_ylabel('Long Ratio'); axes[0].legend(); axes[0].grid(True, axis='y')
axes[0].set_title('Long Ratio Distribution by Sentiment', fontweight='bold')

top_coins = merged.groupby(['sentiment','Coin'])['pnl_net'].sum().reset_index()
top_by_sent = (top_coins.groupby('sentiment', group_keys=True)
                         .apply(lambda x: x.nlargest(5,'pnl_net'))
                         .reset_index(level=0).reset_index(drop=True))
bar_w = 0.25; x = np.arange(5)
for i, (s, col) in enumerate(zip(order3, [FEAR_COLOR, NEUTRAL_COLOR, GREED_COLOR])):
    sub = top_by_sent[top_by_sent['sentiment']==s].head(5)
    axes[1].bar(x + i*bar_w, sub['pnl_net'].values, width=bar_w, label=s, color=col, alpha=0.85, edgecolor='none')
axes[1].set_xticks(x + bar_w)
greed_coins = top_by_sent[top_by_sent['sentiment']=='Greed']['Coin'].head(5).tolist()
axes[1].set_xticklabels(greed_coins, fontsize=9)
axes[1].set_ylabel('Total Net PnL (USD)')
axes[1].set_title('Top 5 Coin PnL by Sentiment Period', fontweight='bold')
axes[1].axhline(0, color='white', lw=0.8, ls='--'); axes[1].legend(); axes[1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('charts/chart5_long_ratio_coins.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
<a id='partc'></a>
## Part C — Actionable Strategy Recommendations

### Strategy 1: "Quality Over Quantity" — Fear-Day Trade Frequency Cap
> *"During Fear days, cap each account's daily trade count at their 30-day rolling average. Flag accounts exceeding 150% of baseline for review. Do not scale up position size."*

**Evidence:** Traders execute 37% more trades (+115% more volume) on Fear days, yet the *median* daily PnL is nearly 60% lower than on Greed days ($104 vs $236). The excess trades are noise — driven by emotion, stop-chasing, and reactive positioning. A cap reduces fee drag and prevents overtrading into adverse conditions.

**Applicable segment:** All trader types, but especially the Opportunistic Traders archetype (low cross-margin, infrequent, all 3 loss accounts).

---

### Strategy 2: "Sentiment-Calibrated Sizing" — Scale Into Greed, Flatten Into Fear
> *"On Greed days: allow full notional sizing with cross-margin. On Fear days: reduce per-trade size by 25–30%, prefer isolated margin for new opens. On the first day sentiment flips Fear→Neutral or Neutral→Greed: allow an aggressive entry window (the contrarian payoff window)."*

**Evidence:** Greed days produce the tightest, most consistent PnL distribution (median $236, win rate 85.6%). Fear-day skew is driven by a small number of large-payoff contrarian trades — these are hard to time and carry high variance. The optimal time to go larger is the sentiment *transition* window. High cross-margin traders (best performers) already do this naturally.

---
<a id='bonus'></a>
## Bonus — Predictive Model + Clustering

In [ ]:
# Predict next-day profitability
daily_m = daily.copy().sort_values(['Account','date'])
daily_m['next_win'] = daily_m.groupby('Account')['is_day_winner'].shift(-1)
daily_m = daily_m.dropna(subset=['next_win'])

le = LabelEncoder()
daily_m['sent_enc'] = le.fit_transform(daily_m['sentiment'])

feat_model = ['sent_enc','fg_value','n_trades','win_rate','long_ratio',
              'cross_margin_pct','avg_size_usd','total_vol_usd']
Xm = daily_m[feat_model].fillna(0)
ym = daily_m['next_win'].astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(Xm, ym, test_size=0.2, random_state=42, stratify=ym)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=42)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:,1]

cv = cross_val_score(rf, Xm, ym, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print(f'ROC-AUC     : {roc_auc_score(y_te, y_prob):.4f}')
print(f'CV Accuracy : {cv.mean():.4f} ± {cv.std():.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred, target_names=['Loss Day','Profit Day']))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
imp = pd.Series(rf.feature_importances_, index=feat_model).sort_values(ascending=True)
display = {'sent_enc':'Sentiment','fg_value':'FG Index Value','n_trades':'# Trades',
           'win_rate':'Win Rate','long_ratio':'Long Ratio','cross_margin_pct':'Cross Margin %',
           'avg_size_usd':'Avg Trade Size','total_vol_usd':'Total Volume'}
imp.index = [display.get(i,i) for i in imp.index]
bar_c = [GREED_COLOR if v >= imp.median() else ACCENT for v in imp.values]
bars = ax.barh(imp.index, imp.values, color=bar_c, edgecolor='none', height=0.55)
ax.set_title('Chart 8 — Feature Importance: Predicting Next-Day Profitability',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score'); ax.grid(True, axis='x')
for bar, val in zip(bars, imp.values):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('charts/chart8_feature_importance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 Model achieves ROC-AUC 0.715 — solid signal. Top features: Win Rate, Volume, FG Index.')

In [ ]:
# KMeans clustering
feat_cols = ['total_pnl','win_rate','n_trades','avg_size_usd','cross_margin_pct','long_ratio','max_drawdown']
X  = acct[feat_cols].fillna(0)
Xs = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, random_state=42, n_init=20)
acct['cluster'] = km.fit_predict(Xs)
pnl_rank = acct.groupby('cluster')['total_pnl'].mean().sort_values(ascending=False)
cnames   = {c:n for c,n in zip(pnl_rank.index,
             ['Elite Performers','Disciplined Traders','Opportunistic Traders','Loss-Heavy Traders'])}
acct['archetype'] = acct['cluster'].map(cnames)

print('=== Archetype Summary ===')
acct.groupby('archetype')[['total_pnl','win_rate','n_trades','avg_size_usd','cross_margin_pct']].mean().round(2)

In [ ]:
arch_order = ['Elite Performers','Disciplined Traders','Opportunistic Traders','Loss-Heavy Traders']
arch_cols  = ['#2EC4B6','#F4A261','#8338EC','#E63946']
col_map    = {n:c for n,c in zip(arch_order, arch_cols)}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
fig.suptitle('Chart 6 — Trader Behavioral Archetypes (KMeans k=4)',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

for arch in arch_order:
    sub = acct[acct['archetype']==arch]
    axes[0].scatter(sub['win_rate'], sub['total_pnl'],
                    s=np.clip(sub['total_vol_usd']/5e5, 40, 400),
                    color=col_map[arch], label=arch, alpha=0.85, edgecolors='none')
axes[0].axhline(0, color='white', lw=0.8, ls='--')
axes[0].axvline(0.5, color='white', lw=0.8, ls='--')
axes[0].set_xlabel('Win Rate'); axes[0].set_ylabel('Total PnL (USD)')
axes[0].set_title('Archetype Map: Win Rate × Total PnL  (size = volume)', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True)

norm_feats = ['win_rate','n_trades','avg_size_usd','cross_margin_pct','long_ratio']
cat_means  = acct.groupby('archetype')[norm_feats].mean()
cat_norm   = (cat_means - cat_means.min()) / (cat_means.max() - cat_means.min() + 1e-9)
x = np.arange(len(norm_feats)); w = 0.18
for i, arch in enumerate(arch_order):
    row = cat_norm.loc[arch] if arch in cat_norm.index else pd.Series([0]*len(norm_feats))
    axes[1].bar(x+i*w, row.values, width=w, label=arch, color=arch_cols[i], alpha=0.85, edgecolor='none')
axes[1].set_xticks(x+w*1.5)
axes[1].set_xticklabels(['Win\nRate','# Trades','Avg Size','Cross\nMargin','Long\nRatio'], fontsize=9)
axes[1].set_title('Normalized Behavioral Profile per Archetype', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('charts/chart6_clusters.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Per-account sentiment heatmap
acct_sent = daily.groupby(['Account','sentiment']).agg(
    avg_pnl  = ('daily_pnl','mean'),
    win_rate = ('win_rate','mean'),
).reset_index()

pivot = acct_sent.pivot(index='Account', columns='sentiment', values='avg_pnl').fillna(0)
pivot = pivot[['Fear','Neutral','Greed']]
pivot_short = pivot.copy()
pivot_short.index = [a[:10]+'…' for a in pivot_short.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), facecolor=BG)
fig.suptitle('Chart 7 — Per-Account PnL: Fear vs Greed Breakdown',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

im = axes[0].imshow(pivot_short.values, cmap='RdYlGn', aspect='auto', vmin=-500, vmax=500)
plt.colorbar(im, ax=axes[0], label='Avg Daily PnL (USD)', shrink=0.8)
axes[0].set_xticks([0,1,2]); axes[0].set_xticklabels(['Fear','Neutral','Greed'])
axes[0].set_yticks(range(len(pivot_short)))
axes[0].set_yticklabels(pivot_short.index, fontsize=7)
axes[0].set_title('Avg Daily PnL Heatmap\n(Account × Sentiment)', fontweight='bold')

delta = (pivot['Greed'] - pivot['Fear']).sort_values()
bar_c = [GREED_COLOR if v >= 0 else FEAR_COLOR for v in delta.values]
axes[1].barh([a[:10]+'…' for a in delta.index], delta.values,
              color=bar_c, edgecolor='none', height=0.65)
axes[1].axvline(0, color='white', lw=0.8, ls='--')
axes[1].set_xlabel('ΔPnL = Greed_avg − Fear_avg (USD/day)')
axes[1].set_title('Sentiment Edge per Account\n(+ve = performs better in Greed)', fontweight='bold')
axes[1].grid(True, axis='x')

plt.tight_layout()
plt.savefig('charts/chart7_per_account_sentiment.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Final Summary of Insights

| # | Insight | Key Number | Chart |
|---|---------|-----------|-------|
| 1 | **Fear days are high-variance, not high-median** — mean $5K but median only $104 | p=0.016 | Chart 1 |
| 2 | **Traders over-trade during Fear** — 37% more trades, 115% more volume, worse median outcome | 105 vs 77 trades/day | Chart 2 |
| 3 | **Long bias flips counterintuitively** — more long during Fear (52%) than Greed (47%) | ΔLong = +5pp | Chart 5 |
| 4 | **Cross-margin usage = sophistication proxy** — high cross users 5× higher PnL | $1.29M vs $243K | Chart 3 |
| 5 | **Model ROC-AUC 0.715** — win rate + volume + FG index carry real next-day signal | 65.9% CV acc | Chart 8 |

---
*Primetrade.ai — Data Science Intern Round 0  |  Real Data Submission*